# main

> The FastHTML app: pages, routes and the taught-sound library.

In [ ]:
#| default_exp main

In [ ]:
#| hide
from nbdev.showdoc import *

Run it with `pixi run serve` (the `nadayantra` command). The app listens on <http://127.0.0.1:5001>; the JSON API for the Observable frontend is mounted from `nadayantra.api`.

In [ ]:
#| export
from fasthtml.common import *
from fastlite import database
import json, random, time, unicodedata, numpy as np
from nadayantra.core import repo_root
from nadayantra.engine import nada, pcm, trim, dim
from nadayantra import keys

In [ ]:
#| export
root = repo_root()
(root/'data').mkdir(exist_ok=True)
app,rt = fast_app(hdrs=(Link(rel='stylesheet', href='/style.css'), Link(rel='icon', href='/icon.png'), Script(src='/app.js')),
                  static_path=str(root/'static'), key_fname=str(root/'.sesskey'), on_startup=[nada.warm_up])

## glyphs: [symbol, meaning, devanagari, iast], in four sections

In [ ]:
#| export
kinds = dict(functions='Functions', monadic_operators='Monadic operators', dyadic_operators='Dyadic operators', syntax_and_literals='Syntax and literals')
table = json.loads((root/'glyph_table.json').read_text())
G,SECS = {},[]
for k,title in kinds.items():
    gs = [dict2obj(dict(sym=s, mean=m, dev=d, iast=i, sec=title)) for s,m,d,i in table[k]]
    gs = [g for g in gs if G.setdefault(g.sym, g) is g]  # a repeated symbol keeps its first row
    if gs: SECS.append((title, gs))

## what a sound types: a few Unicode characters, or a named special key

In [ ]:
#| export
SP = {'space':'␣ Space', 'return':'⏎ Return', 'tab':'⇥ Tab', 'backspace':'⌫ Backspace', 'escape':'⎋ Escape',
      'left':'← Left', 'right':'→ Right', 'up':'↑ Up', 'down':'↓ Down'}
bad = {'Cc','Cf','Cn','Co','Cs'}
def typable(t): return 0 < len(t) <= 8 and not any(c.isspace() or unicodedata.category(c) in bad for c in t)
def typed(sym): return sym.split(',')[0].replace('…','').strip() or sym  # "(…)" types "()", "₁, ₋₁, …" types "₁"
def lab(k): return SP.get(k, k)
def cps(k): return '' if k in SP else ' '.join(f'U+{ord(c):04X}' for c in k)

## taught sounds: a name and up to 12 embeddings (float32 blob) each

In [ ]:
#| export
class Sound: id:int=None; name:str=''; glyph:str=''; key:str=''; samples:bytes=b''
db = database(str(root/'data'/'gnn.db'))
sounds = db.create(Sound, pk='id')

In [ ]:
#| export
def blob(x): return np.asarray(x, 'f4').reshape(-1, dim)[-12:].tobytes()
def vecs(s): return np.frombuffer(s.samples, 'f4').reshape(-1, dim)
def snd(sym): return first(s for s in sounds() if s.glyph==sym)
def skey(s): return s.key if s.key in SP or typable(s.key) else (typed(s.glyph) if s.glyph in G else '')
def add_ex(s, e): sounds.update(id=s.id, samples=blob(np.vstack([vecs(s), e])))

In [ ]:
#| export
def newname(taken):
    "A made-up name like rum, row or reh"
    on,vo,co = 'r b d f g h j k l m n p s t v z sh ch'.split(), 'a e i o u eh ow ee oo ah'.split(), list('mnktpsglbd')
    for _ in range(500):
        v = random.choice(vo)
        n = random.choice(on) + v + random.choice(co if len(v)<2 else ['']+co)  # two-letter vowels already end a word nicely
        if n not in taken: return n
    return f'snd{len(taken)+1}'

In [ ]:
#| export
# Below this cosine the closest sound is probably not what you made: a speaker repeating a sound
# averaged .82 (lowest .27), a different sound .40 (95th percentile .72). Tune it on your voice.
mincos = .6
st = AttrDict(rec=[], hist=[])  # last recognition, and what the keyboard typed

In [ ]:
#| export
def rank(e, keyed=False):
    ss = [s for s in sounds() if not keyed or skey(s)]
    cs = [float((vecs(s)@e).max()) for s in ss]
    if not cs: return []
    w = np.exp((np.array(cs)-max(cs))*40)  # sharpen: audio-to-audio cosines sit close together
    return sorted([AttrDict(id=s.id, name=s.name, glyph=s.glyph, cos=c, share=float(x/w.sum())) for s,c,x in zip(ss,cs,w)], key=lambda m: -m.cos)

In [ ]:
#| export
def hit(): return st.rec[0].id if st.rec and st.rec[0].cos >= mincos else None

In [ ]:
#| export
async def embed(audio):
    x = trim(pcm(audio))
    if x is None: return None, "I didn't hear anything. Check the microphone and try again."
    try: return await nada.embed(x), None
    except Exception as e: return None, str(e)

In [ ]:
#| export
# carry over what the macOS app has taught: same model, same embeddings
mac = Path.home()/'Library/Application Support/alex-jacob.gananayantra/sound-library.json'
if not sounds() and mac.exists():
    for r in json.loads(mac.read_text()):
        x = [s for s in r.get('samples',[]) if len(s)==dim]
        if x and r.get('name'): sounds.insert(Sound(name=r['name'], glyph=r.get('glyph') or '', key=r.get('key') or '', samples=blob(x)))

## page

In [ ]:
#| export
def hue(i, n): return f'--t:{i/max(n-1,1):.3f}'  # yellow at the first item, red at the last
def NStat():
    s = nada.state
    kw = dict(hx_get='/nstat', hx_trigger='every 2s', hx_swap='outerHTML') if s in ('idle','starting') else {}
    msg = dict(idle='', starting='Loading nada…', ready=f'nada ready ({nada.device})', failed='nada unavailable')[s]
    return Small(msg, id='nstat', aria_busy='true' if s=='starting' else None, **kw)

In [ ]:
#| export
def Splash():
    s = nada.state
    if s == 'ready': return Div(id='splash')
    if s == 'failed': body = (H3('nada couldn’t start'), P(nada.error, cls='err'), Div(
        Button('Try Again', hx_post='/retry', hx_target='#splash', hx_swap='outerHTML'), Button('Continue Anyway', cls='secondary', onclick='skip()'), role='group'))
    else: body = (P('Preparing nada', aria_busy='true'), Small('Loading the sound model. The first launch can take a minute.'))
    kw = dict(hx_get='/splash', hx_trigger='every 1s', hx_swap='outerHTML') if s != 'failed' else {}
    return Div(Img(src='/icon.png', width=112), *body, id='splash', cls='splash', **kw)

In [ ]:
#| export
HERO = dict(keys=('Speak in notations.', 'Hold a key, make a sound, and its glyph is typed.'),
            names=('Teach it a sound.', 'Make a sound for each glyph, and it will remember.'))

In [ ]:
#| export
def page(cur, *c):
    tabs = [('keys','/','Sound Keyboard'), ('names','/names','Sound Names')]
    return Title('shabdayantra'), Div(Splash(),
        Nav(Ul(Li(Img(src='/icon.png', height=28), Strong(' shabdayantra'))),
            Ul(*[Li(A(t, href=h, aria_current='page' if k==cur else None)) for k,h,t in tabs], Li(NStat()))),
        Header(H1(HERO[cur][0]), P(HERO[cur][1]), cls='hero'), Main(*c), cls='container')

In [ ]:
#| export
@rt
def nstat(): return NStat()
@rt
def splash(): return Splash()
@rt
def retry(): nada.warm_up(); return Splash()

## key picker: type any characters, or choose a special key; blank means "type the glyph"

In [ ]:
#| export
def ix(s): ids = [x.id for x in sounds()]; return ids.index(s.id), len(ids)
R = dict(det=lambda s: Detail(s.glyph), row=lambda s: Row(s, *ix(s)), kbd=lambda s: (KRow(s, *ix(s)), KMeta(True)))

In [ ]:
#| export
def KeyPick(s, ctx):
    gk = typed(s.glyph) if s.glyph in G else ''
    v,tgt = dict(id=s.id, ctx=ctx), dict(det='#det', row=f'#r{s.id}', kbd=f'#k{s.id}')[ctx]
    kw = dict(name='v', hx_post='/setkey', hx_trigger='change', hx_target=tgt, hx_swap='outerHTML')
    return Span(Input(value='' if s.key in SP else s.key, placeholder=gk or 'Key', maxlength=8, hx_vals=dict(v, f='t'), **kw),
                Select(Option(f'Glyph ({gk})' if gk else 'No key', value='-'), *[Option(l, value=n, selected=s.key==n) for n,l in SP.items()], hx_vals=dict(v, f='s'), **kw), cls='kp')

In [ ]:
#| export
@rt
def setkey(id:int, f:str, v:str='', ctx:str='row'):
    s,v = sounds[id], v.strip()
    if f=='s': k = '' if v=='-' else v
    elif not v: k = s.key if s.key in SP else ''
    else: k = v if typable(v) and v not in SP else s.key  # "up" typed as text is a word, not the Up arrow
    sounds.update(id=id, key=k)
    return R[ctx](sounds[id])

## Sound Names: teach a sound per glyph, make it again to get the name back

In [ ]:
#| export
def Notice(msg='', err=False): return Div(msg, id='notice', cls='err' if err else 'ok', hx_swap_oob='true')

In [ ]:
#| export
def Recog():
    r = st.rec
    if not r: return Div(id='recog', hx_swap_oob='true')
    b = r[0]
    top = H2(b.glyph, ' ', b.name, Small(f' {b.share:.0%}')) if b.cos >= mincos else (H3('Not sure', cls='warn'), P(f'That doesn’t sound much like anything taught. The closest is “{b.name}”.'))
    return Article(top, *[Div(Span(f'{m.glyph} {m.name}'), Progress(value=m.share, max=1), Small(f'cosine {m.cos:.3f}'), cls='mrow') for m in r[:5]], id='recog', hx_swap_oob='true')

In [ ]:
#| export
def done(msg='', err=False, view='g', sel=''): return Lib(view, sel), Notice(msg, err), Recog()

In [ ]:
#| export
def Tile(g, i, n, sel, s):
    cls = 'match' if s and s.id==hit() else 'taught' if s else ''
    return Div(Small(g.mean), B(g.sym), I(g.iast), cls=f'tile {cls} {"sel" if g.sym==sel else ""}', style=hue(i,n), role='button', tabindex=0,
               data_sym=g.sym, data_iast=g.iast, data_taught=int(bool(s)), title=f'{g.sym}  {g.mean}  {g.dev}  {g.iast}')

In [ ]:
#| export
def Detail(sym=''):
    g,s = G.get(sym), snd(sym)
    if not g: return Div(id='det')
    n = len(vecs(s)) if s else 0
    acts = (Small(f'{n} example{"s"[:n!=1]}'), Small(f'types {lab(skey(s))}', Br(), Code(cps(skey(s)))), KeyPick(s, 'det'),
            Button('Add Example', data_rec='/more', data_id=s.id, data_hint='Make the same sound again, then press Stop.'),
            Button('Forget', cls='outline', hx_post='/forget', hx_vals=dict(id=s.id, view='g', sel=sym), hx_target='#lib', hx_swap='outerHTML')
           ) if s else (Button('Teach Sound', data_rec='/learn', data_sym=sym, data_hint=f'Make the sound for {sym} ({g.iast}), then press Stop.'),)
    return Div(Div(g.sym, cls='big'), Div(Strong(f'{g.iast}  {g.dev}'), Br(), g.mean, Br(), Small(g.sec)), Div(*acts, cls='acts'), id='det')

In [ ]:
#| export
def Row(s, i, n):
    return Div(Span('✓' if s.id==hit() else '~'), Span(s.glyph), Input(value=s.name, name='v', hx_post='/rename', hx_vals=dict(id=s.id), hx_trigger='change', hx_swap='none'),
               Small(f'{len(vecs(s))} ex.'), KeyPick(s, 'row'),
               Button('+', title='Record another example', data_rec='/more', data_id=s.id, data_hint='Make the same sound again, then press Stop.'),
               Button('×', cls='outline', title='Forget this sound', hx_post='/forget', hx_vals=dict(id=s.id, view='l'), hx_target='#lib', hx_swap='outerHTML'),
               id=f'r{s.id}', cls='row', style=hue(i,n))

In [ ]:
#| export
def Lib(view='g', sel=''):
    tabs = Div(*[Button(t, cls='' if view==v else 'outline', hx_get='/lib', hx_vals=dict(view=v, sel=sel), hx_target='#lib', hx_swap='outerHTML') for v,t in (('g','Glyphs'), ('l','All sounds'))], role='group', cls='tabs')
    if view == 'g':
        by = {s.glyph:s for s in sounds() if s.glyph}
        body = (*[Div(H4(t, Small(f'  {sum(g.sym in by for g in gs)} of {len(gs)} taught')),
                      Div(*[Tile(g, i, len(gs), sel, by.get(g.sym)) for i,g in enumerate(gs)], cls='tiles')) for t,gs in SECS], Detail(sel))
    else:
        ss = sounds()
        body = (H4(f'Taught sounds ({len(ss)})', Button('Remove All', cls='outline', hx_post='/forgetall', hx_target='#lib', hx_swap='outerHTML', hx_confirm='Forget every sound?')),
                *[Row(s, i, len(ss)) for i,s in enumerate(ss)]) if ss else (P('No sounds taught yet. Press Teach a New Sound and make a sound.'),)
    return Div(tabs, *body, id='lib', data_view=view, data_sel=sel)

In [ ]:
#| export
@rt
def lib(view:str='g', sel:str=''): return Lib(view, sel)
@rt
def detail(sym:str): return Detail(sym)

In [ ]:
#| export
@rt('/names')
def names():
    return page('names',
        Div(Div(Button('Teach a New Sound', cls='rb', data_rec='/teach', data_hint='Make the sound, then press Stop.'),
                Small('types'), Input(id='tk', maxlength=8, placeholder='Key'),
                Select(Option('Text', value=''), *[Option(l, value=n) for n,l in SP.items()], id='ts'),
                Button('Recognize', cls='rb secondary', data_rec='/recognize', data_hint='Make a sound, then press Stop.'), id='idle'),
            Div(Button('Stop', cls='contrast', onclick='stop()'), Progress(cls='meter', value=0, max=1), Small('0.0 / 10 s', cls='secs'), Small(id='hint'), id='live')),
        Div(id='notice'), Div(id='recog'), Lib())

In [ ]:
#| export
async def learned(audio, view, sel, f):
    e,err = await embed(audio)
    return done(err, True, view, sel) if err else done(f(e), False, view, sel)

In [ ]:
#| export
@rt
async def teach(audio:str, key:str='', special:str='', view:str='g', sel:str=''):
    k = special if special in SP else key.strip() if typable(key.strip()) and key.strip() not in SP else ''
    def f(e):
        s = sounds.insert(Sound(name=newname({x.name for x in sounds()}), key=k, samples=blob(e)))
        return f'Saved as “{s.name}”' + (f' (types {lab(k)})' if k else '') + '. Make the same sound again and press Recognize.'
    return await learned(audio, view, sel, f)

In [ ]:
#| export
@rt
async def learn(audio:str, sym:str, view:str='g', sel:str=''):
    def f(e):
        if s := snd(sym): add_ex(s, e); return f'Added another example to {sym} “{s.name}”.'
        s = sounds.insert(Sound(name=G[sym].iast, glyph=sym, samples=blob(e)))
        return f'Saved {sym} as “{s.name}”. Make the same sound again and press Recognize.'
    return await learned(audio, view, sel, f)

In [ ]:
#| export
@rt
async def more(audio:str, id:int, view:str='g', sel:str=''):
    def f(e): add_ex(sounds[id], e); return f'Added another example to “{sounds[id].name}”.'
    return await learned(audio, view, sel, f)

In [ ]:
#| export
@rt
async def recognize(audio:str, view:str='g', sel:str=''):
    if not sounds(): return done('Teach at least one sound first.', True, view, sel)
    e,err = await embed(audio)
    if err: return done(err, True, view, sel)
    st.rec = rank(e)
    return done('', False, view, sel)

In [ ]:
#| export
@rt
def rename(id:int, v:str): sounds.update(id=id, name=v)

In [ ]:
#| export
@rt
def forget(id:int, view:str='g', sel:str=''): sounds.delete(id); st.rec = []; return done('', False, view, sel)
@rt
def forgetall(): [sounds.delete(s.id) for s in sounds()]; st.rec = []; return done('', False, 'l', '')

## Sound Keyboard: hold a key, make a sound, let go; the matching sound types its key.

A web page cannot type into other apps, so the keys land in the box on this page.

In [ ]:
#| export
def KMeta(oob=False): return Div(id='kmeta', hidden=True, data_n=sum(bool(skey(s)) for s in sounds()), **(dict(hx_swap_oob='true') if oob else {}))
def Msg(t, k=''): return Div(t, id='kst', cls=k)

In [ ]:
#| export
def KRow(s, i, n):
    k = skey(s)
    return Div(Span(s.glyph), Strong(s.name), Small(cps(k)), Small(f'{len(vecs(s))} ex.'), KeyPick(s, 'kbd'), id=f'k{s.id}', cls='row', style=hue(i,n)+('' if k else ';opacity:.55'))

In [ ]:
#| export
def Log(oob=False):
    h = st.hist
    return Div(Div(Strong('Typed'), Button('Clear', cls='outline', hx_post='/clear', hx_target='#log', hx_swap='outerHTML'), cls='hd') if h else P('Nothing typed yet. Each sound you make will show up here.'),
               *[Div(Code(lab(x.k)), Small(x.name), Small(cps(x.k)), Small(f'{x.cos:.2f}'), cls='row', style=hue(i,len(h))) for i,x in enumerate(h)],
               id='log', **(dict(hx_swap_oob='true') if oob else {}))

In [ ]:
#| export
@rt('/')
def index():
    ss = sounds()
    return page('keys',
        Div(Label(Div(Strong('Sound keyboard'), Small('Turn it on, then hold the key and make a sound.')), Input(type='checkbox', role='switch', id='kon'), cls='ctl'),
            Div(Div(Strong('Hold to talk'), Small('The key you hold while making a sound.')),
                Select(Option('Right Option ⌥', value='AltRight'), Option('Right Command ⌘', value='MetaRight'), Option('Right Control ⌃', value='ControlRight'), id='pk'), cls='ctl'),
            Label(Div(Strong('Use in other apps'), Small('Types into the app in front and watches the key everywhere. macOS asks you to allow it.')),
                  Input(type='checkbox', role='switch', id='sys'), cls='ctl') if keys.available else '', cls='panel'),
        Msg('Off. Nothing is typed.'), KMeta(),
        Div(Button('Hold to Talk', id='hold', cls='secondary', disabled=True), Small('The same as holding the key, for trying it here.'), cls='hold'),
        Textarea(id='scratch', rows=3, placeholder='Click here, then try it: characters appear in this box'),
        Div(Button('Clear', cls='outline', onclick="$('#scratch').value=''"), cls='under'),
        Div(Div(H4('Sounds and what they type'), *[KRow(s, i, len(ss)) for i,s in enumerate(ss)]) if ss else P('No sounds yet. Teach a sound for each glyph on the Sound Names page. Each one types its own glyph.'),
            Log(), cls='stack'))

In [ ]:
#| export
@rt('/sys/status')
def sysstatus(): return keys.status()
@rt('/sys/prompt')
def sysprompt(): keys.prompt(); return keys.status()
@rt('/sys/events')
def sysevents():
    "Tells the page when the hold key goes down or up in any app. Needs Input Monitoring."
    if not keys.watch(): return Response('Input Monitoring is off', status_code=403)
    return StreamingResponse(keys.events(), media_type='text/event-stream', headers={'Cache-Control':'no-cache', 'X-Accel-Buffering':'no'})

In [ ]:
#| export
@rt
async def typekey(audio:str, sys:bool=False):
    if not any(skey(s) for s in sounds()): return Msg('No sound has a key yet. Assign keys on the Sound Keyboard page, or teach a glyph.', 'warn'), Log(True)
    e,err = await embed(audio)
    if err: return Msg(err, 'err'), Log(True)
    b = rank(e, keyed=True)[0]
    if b.cos < mincos: return Msg(f'Not sure. Closest was “{b.name}” ({b.cos:.2f}), so nothing was typed.', 'warn'), Log(True)
    k = skey(sounds[b.id])
    st.hist.insert(0, AttrDict(k=k, name=b.name, cos=b.cos)); del st.hist[30:]
    if sys:  # into whichever app has focus; the page's own box is left alone, or it would type twice
        if keys.type_key(k): return Msg(f'Typed {lab(k)} for “{b.name}” into the active app.', 'ok'), Log(True)
        return Msg('Typing into other apps is off in macOS. Turn the switch off and on again to fix it.', 'err'), Log(True)
    return HtmxResponseHeaders(trigger=json.dumps(dict(typed=dict(k=k)))), Msg(f'Typed {lab(k)} for “{b.name}”.', 'ok'), Log(True)

In [ ]:
#| export
@rt
def clear(): st.hist.clear(); return Log()

In [ ]:
#| export
from nadayantra.api import mount
mount(app, rt, globals())  # JSON for the Observable frontend, and its build at /app/. Nothing above changes.

In [ ]:
#| export
def main():
    "Run the web app on http://127.0.0.1:5001"
    import uvicorn
    uvicorn.run(app, host='127.0.0.1', port=5001)